In [1]:
from pathlib import Path
import sys
import subprocess
import time
import importlib.metadata
import json
import pandas as pd
from IPython.display import display

In [2]:
ROOT = Path.cwd()
TASKS = ['a_import','b_prepare','c_fit','d_simulate','e_diagnose','f_plot','g_report']
# recreate ignored task directories and input links on a fresh checkout
for task in TASKS:
    (ROOT/task/'output').mkdir(parents=True,exist_ok=True)
(ROOT/'a_import/input').mkdir(parents=True,exist_ok=True)
links = {f'{task}/input':f'../{previous}/output' for previous,task in zip(TASKS,TASKS[1:])}
links.update({'c_fit/output/data':'../input','d_simulate/output/fit':'../input','e_diagnose/output/simulation':'../input'})
for relative,target in links.items():
    path = ROOT/relative
    if not path.is_symlink():path.symlink_to(target,target_is_directory=True)
plan = [path for task in TASKS for path in sorted((ROOT/task/'src').rglob('*.ipynb'))]
pd.DataFrame({'order':range(1,len(plan)+1),'notebook':[str(p.relative_to(ROOT)) for p in plan]})

,order,notebook
0,1,a_import/src/import.ipynb
1,2,b_prepare/src/prepare.ipynb
2,3,c_fit/src/manuscript/02_lognormal_grw.ipynb
3,4,c_fit/src/manuscript/03_ar1_s.ipynb
4,5,c_fit/src/manuscript/04_hurdle_ar1_s_p.ipynb
5,6,c_fit/src/manuscript/06_se_hurdle_s.ipynb
6,7,c_fit/src/supplement/01_arw4.ipynb
7,8,c_fit/src/supplement/05_ar6_6.ipynb
8,9,c_fit/src/supplement/07_lognormal_grw_s.ipynb
9,10,c_fit/src/supplement/08_ar1_y.ipynb


In [3]:
# isolated processes, with notebook source left unchanged
WORKER = r"""import json, os, sys
from pathlib import Path
import matplotlib
from IPython.core.interactiveshell import InteractiveShell
from IPython.utils.capture import capture_output

path = Path(sys.argv[1]).resolve()
os.chdir(path.parent)
matplotlib.use('Agg')
notebook = json.loads(path.read_text())
shell = InteractiveShell.instance()
for index, cell in enumerate(notebook['cells'], 1):
    with capture_output() as captured:
        result = shell.run_cell(''.join(cell['source']), store_history=False)
    error = result.error_before_exec or result.error_in_exec
    if error:
        print(f'Cell {index}:\n{captured.stdout}{captured.stderr}', file=sys.stderr)
        raise error
"""
runs = []
for path in plan:
    relative = str(path.relative_to(ROOT))
    start = time.perf_counter()
    result = subprocess.run([sys.executable, '-c', WORKER, str(path)], capture_output=True, text=True)
    status = 'passed' if result.returncode == 0 else 'failed'
    runs.append({'notebook':relative, 'seconds':time.perf_counter()-start, 'status':status})
    pd.DataFrame(runs).to_csv(ROOT/'pipeline_run.csv',index=False)
    if result.returncode:
        raise RuntimeError(f'{relative}\n{result.stdout}\n{result.stderr}')
display(pd.DataFrame(runs))

,notebook,seconds,status
0,a_import/src/import.ipynb,1.313504,passed
1,b_prepare/src/prepare.ipynb,1.416911,passed
2,c_fit/src/manuscript/02_lognormal_grw.ipynb,1.877329,passed
3,c_fit/src/manuscript/03_ar1_s.ipynb,1.873032,passed
4,c_fit/src/manuscript/04_hurdle_ar1_s_p.ipynb,1.778798,passed
5,c_fit/src/manuscript/06_se_hurdle_s.ipynb,7.932033,passed
6,c_fit/src/supplement/01_arw4.ipynb,1.930479,passed
7,c_fit/src/supplement/05_ar6_6.ipynb,2.272356,passed
8,c_fit/src/supplement/07_lognormal_grw_s.ipynb,0.938715,passed
9,c_fit/src/supplement/08_ar1_y.ipynb,0.916694,passed


In [4]:
packages = ['numpy','pandas','scipy','scikit-learn','matplotlib','seaborn','ipython']
versions = pd.DataFrame([{'package':p,'version':importlib.metadata.version(p)} for p in packages])
versions.to_csv(ROOT/'environment_versions.csv',index=False)
display(versions)

,package,version
0,numpy,2.5.1
1,pandas,3.0.3
2,scipy,1.18.0
3,scikit-learn,1.9.0
4,matplotlib,3.11.0
5,seaborn,0.13.2
6,ipython,9.15.0
